In [57]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import datetime

In [130]:
# we are missing all urc (regular season and knockout) matches from the 2024 part of the 2023/24 season
# we are also missing the champions cup matches from that part of the season

In [183]:
urc_teams = ["stormers", "leinster", "ulster", "gwent-dragons", "glasgow", "sharks", "bulls", "ospreys", "zebre", "edinburgh", "scarlets", "munster", "cardiff-blues", "lions", "connacht", "benetton"]
prem_teams = ["sale", "gloucester", "harlequins", "bath", "newcastle", "saracens", "northampton", "exeter-chiefs", "bristol", "leicester"]
top14_teams = ["stade-francais", "us-montauban", "perpignan", "bayonne", "castres", "pau", "lyon", "racing-92", "montpellier", "toulon", "bordeaux", "la-rochelle", "clermont", "toulouse"]

In [132]:
def make_fixtures_url(team):
    return f"https://www.rugbypass.com/teams/{team}/fixtures-results/"

In [134]:
# we should extract ALL matches (i.e. URC and CC) from these teams in 2024 until around 1st September (URC began 20th September)

In [28]:
def retreive_fixtures(year, url):

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Origin": "https://www.rugbypass.com",
        "Referer": "https://www.rugbypass.com/",
    })

    payload = {
            "action": "get-fixtures",
            "season": f"{year}",
            "isContent": "1",
        }
    
    response = session.post(url, files={k: (None, v) for k, v in payload.items()})
    data = response.json()
    
    return data

In [58]:
def parse_date(s):
    wd, day, m, y = s.split()
    month_dict = {
        "Jan": 1,
        "Feb": 2, 
        "Mar": 3,
        "Apr": 4,
        "May": 5,
        "Jun": 6,
        "Jul": 7,
        "Aug": 8,
        "Sep": 9,
        "Oct": 10,
        "Nov": 11,
        "Dec": 12
    }
    month = month_dict[m.strip(",")]
    return datetime.date(int(y), month, int(day))

In [170]:
def urls_from_bs(league_id, season, games):
    urls = []
    for g in games: 
        game_date = parse_date(g.select_one(".date").get_text(strip=True))
        #if urc_seasons[season]["start"] <= game_date and game_date <= urc_seasons[season]["end"]:
        competition = g.select_one(".title").get_text(strip=True)
        
        if competition.strip() == "Investec Champions Cup":
            competition_id = 4
        else:
            competition_id = league_id

        try: 
            url = g.select_one(".link-box")["href"]
            _, g_id = url.split("=")
            urls.append({
                "match_id": int(g_id),
                "year": season, 
                "url": url, 
                "competition_id": competition_id, 
                "date": game_date})
        except TypeError:
            print(g)
    return urls

In [171]:
def urls_from_teams(league_id, seasons, teams):

    all_urls = []
    for season in seasons:
        for team in teams:
            data = retreive_fixtures(season, make_fixtures_url(team))
            games = BeautifulSoup(data["fixtures"]).select("div.date-section")
    
            all_urls += urls_from_bs(league_id, season, games)

    return all_urls

In [184]:
new_urls = urls_from_teams(2, [2024, 2025], top14_teams)

In [185]:
deduped = [dict(t) for t in {tuple(d.items()) for d in new_urls}]

In [186]:
columns = ["match_id", "year", "url", "competition_id"]

new_urls = pd.DataFrame(deduped, columns=columns).set_index("match_id")

In [195]:
new_urls.to_parquet("missing_top14.parquet")

In [198]:
tables = [pd.read_parquet(name + ".parquet") for name in ["all_urls", "missing_urc", "missing_top14", "missing_prem"]] 

In [206]:
df = pd.concat(tables)
df = df[~df.index.duplicated(keep="first")]

In [208]:
df

,year,url,competition_id
match_id,,,
921086,2022,https://www.rugbypass.com/live/ireland-vs-wale...,0
921087,2022,https://www.rugbypass.com/live/england-vs-scot...,0
921088,2022,https://www.rugbypass.com/live/france-vs-italy...,0
921089,2022,https://www.rugbypass.com/live/scotland-vs-wal...,0
921090,2022,https://www.rugbypass.com/live/france-vs-irela...,0
...,...,...,...
943623,2025,https://www.rugbypass.com/live/bath-vs-leicest...,3
939027,2024,https://www.rugbypass.com/live/sale-vs-saracen...,3
943604,2025,https://www.rugbypass.com/live/exeter-chiefs-v...,3


In [202]:
df_old = pd.read_parquet("all_urls.parquet")

In [209]:
df.to_parquet("new_urls.parquet")